# Optical_Flow_Sparse_with_background_subtraction_and image_enhancment  and 1$

In [28]:
import cv2 
import math
import numpy as np
from dollarpy import Recognizer, Template, Point

In [29]:
def singleScaleRetinex(img, sigma):

    retinex = np.log10(img) - np.log10(cv2.GaussianBlur(img, (0 ,0), sigma))
    return retinex

def multiScaleRetinex(img, sigma_list):

    retinex = np.zeros_like(img)
    for sigma in sigma_list:
        retinex += singleScaleRetinex(img, sigma)

    retinex = retinex / len(sigma_list)

    return retinex

def simplestColorBalance(img, low_clip, high_clip):    

    total = img.shape[0] * img.shape[1]
    for i in range(img.shape[2]):
        unique, counts = np.unique(img[:, :, i], return_counts=True)
        current = 0
        for u, c in zip(unique, counts):            
            if float(current) / total < low_clip:
                low_val = u
            if float(current) / total < high_clip:
                high_val = u
            current += c
                
        img[:, :, i] = np.maximum(np.minimum(img[:, :, i], high_val), low_val)

    return img    


def MSRCP(img, sigma_list, low_clip, high_clip):

    img = np.float64(img) + 1.0

    intensity = np.sum(img, axis=2) / img.shape[2]    

    retinex = multiScaleRetinex(intensity, sigma_list)

    intensity = np.expand_dims(intensity, 2)
    retinex = np.expand_dims(retinex, 2)

    intensity1 = simplestColorBalance(retinex, low_clip, high_clip)

    intensity1 = (intensity1 - np.min(intensity1)) / \
                 (np.max(intensity1) - np.min(intensity1)) * \
                 255.0 + 1.0

    img_msrcp = np.zeros_like(img)
    
    for y in range(img_msrcp.shape[0]):
        for x in range(img_msrcp.shape[1]):
            B = np.max(img[y, x])
            A = np.minimum(256.0 / B, intensity1[y, x, 0] / intensity[y, x, 0])
            img_msrcp[y, x, 0] = A * img[y, x, 0]
            img_msrcp[y, x, 1] = A * img[y, x, 1]
            img_msrcp[y, x, 2] = A * img[y, x, 2]

    img_msrcp = np.uint8(img_msrcp - 1.0)

    return img_msrcp


# Define 'Template' gestures, each consisting of a name and a list of 'Point' elements.
# These 'Point' elements have 'x' and 'y' coordinates and optionally the stroke index a point belongs to.
# normal and abnoraml (plan c)
tmpl_1 = Template('Left',[Point(63, 368),
 Point(41, 352),
 Point(31, 347),
 Point(20, 340),
 Point(19, 332),
 Point(14, 321),
 Point(21, 318)
])

tmpl_2 = Template('Down',
[
 Point(10, 235),
 Point(18, 234),
 Point(27, 234),
 Point(58, 239),
 Point(64, 245),
 Point(71, 254),
 Point(54, 262),
 Point(59, 268),
 Point(63, 274),
 Point(68, 279),
 Point(72, 283),
 Point(76, 288)
 ])

tmpl_3 = Template('Up',[Point(21, 307),
 Point(25, 305),
 Point(27, 288),
 Point(28, 276),
 Point(29, 268),
 Point(35, 267),
 Point(34, 255),
 Point(36, 245)])

tmpl_4 = Template('Right',[Point(101, 294),
 Point(104, 304),
 Point(107, 311),
 Point(109, 320),
 Point(110, 328),
 Point(113, 331)])


# Create a 'Recognizer' object and pass the created 'Template' objects as a list.
recognizer = Recognizer([tmpl_1, tmpl_2, tmpl_3, tmpl_4])


In [31]:
cap = cv2.VideoCapture('F:/Behavoirs/Nice camera angle/1 (5)_Trim.mp4')
backgroundobject = cv2.createBackgroundSubtractorMOG2(history = 300, varThreshold = 200 ,detectShadows = True)

lower_val = (25, 0, 0)
upper_val = (102,255,255)

lk_params = dict(winSize  = (15, 15),
                maxLevel = 2,
                criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

suc, frame = cap.read()
resized_frame = cv2.resize(frame, None, fx = 0.25, fy = 0.25, interpolation = cv2.INTER_LINEAR) #resize the frame 
old_gray = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY) #to gray

variance = [15,80,250]

traject = []

z = []

count = 1
ct = 0

while cap.isOpened():
    ret, frame = cap.read()
    if ret:
        count += 3
        cap.set(cv2.CAP_PROP_POS_FRAMES, count) #skip frames

        resized_frame = cv2.resize(frame, None, fx = 0.25, fy = 0.25, interpolation = cv2.INTER_LINEAR) #resize the frame
        org_frame = resized_frame.copy() #copy
        resized_frame = cv2.GaussianBlur(resized_frame, (7,7), 0) #blur
        
        imgHSV = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2HSV)#tohsv
        maskr = cv2.inRange(imgHSV, lower_val, upper_val)#to remove green
        sub = backgroundobject.apply(maskr) #background subtraction
        
        contours , hi = cv2.findContours(sub, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)# finding contours
        c = max(contours, key = cv2.contourArea) # max contours 
        x,y,w,h = cv2.boundingRect(c)
        cv2.rectangle(org_frame, (x, y), (x + w, y + h),(0, 255, 255), 1)


        M = cv2.moments(c) #getting the centroid of max contours
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        
        xx = 150
        yy = 150
        
        if(cy - yy > 0 and cx - xx > 0):
            ROI = org_frame[cy-yy:cy+yy , cx-xx:cx+xx] #getting the reoigon of interset 
            img_msr = MSRCP(ROI , variance, 0.1, 0.99) #image enhancement 
        elif(cy - yy > 0):
            ROI = org_frame[cy-yy:cy+yy , cx:cx+xx] #getting the reoigon of interset 
            img_msr = MSRCP(ROI , variance, 0.1, 0.99) #image enhancement
        elif(cx - xx > 0):
            ROI = org_frame[cy:cy+yy , cx-xx:cx+xx] #getting the reoigon of interset 
            img_msr = MSRCP(ROI , variance, 0.1, 0.99) #image enhancement

    #Calculate optical flow using Lucas-Kanade 
        old_pts = np.array([[cx,cy]], dtype="float32").reshape(-1,1,2)
        new_pts,status,err = cv2.calcOpticalFlowPyrLK(old_gray, sub, old_pts, None, **lk_params)
        p0r, _st, _err = cv2.calcOpticalFlowPyrLK(sub, old_gray, new_pts, None, **lk_params)  
        cv2.circle(org_frame, (int(p0r.ravel()[0]) ,int(p0r.ravel()[1])), 2, (0,255,0), 2)
        traject.append((int(p0r.ravel()[0]) ,int(p0r.ravel()[1])))
        cv2.polylines(org_frame, [np.int32(traject[2:]) for i in traject], False, (0, 255, 0))
        old_gray = sub.copy()
        old_pts = p0r.copy()
        
        # Call 'recognize(...)' to match a list of 'Point' elements to the previously defined templates
        if(len(traject) >= 10):
            for j in range(2 , len(traject)):
                x , y = traject[j]
                z.append(Point(x,y))    
            result = recognizer.recognize(z)
            #cv2.putText(org_frame, result[0], (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            #traject.clear()
        cv2.imshow('Optical Flow', org_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    else:
        cap.release()
        break

        
cv2.destroyAllWindows()    

In [ ]:
# cap = cv2.VideoCapture('F:/Behavoirs/Nice camera angle/1 (5)_Trim.mp4')
# backgroundobject = cv2.createBackgroundSubtractorMOG2(history = 500, varThreshold = 16.5 ,detectShadows = True)

# lower_val = (15, 0, 0)
# upper_val = (130,255,255)

# low_green = np.array([25, 2, 1])
# high_green = np.array([102, 255, 255])

# lk_params = dict(winSize  = (15, 15),
#                 maxLevel = 2,
#                 criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# suc, frame = cap.read()
# resized_frame = cv2.resize(frame, None, fx = 0.25, fy = 0.25, interpolation = cv2.INTER_LINEAR)# resize the frame 
# old_gray = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY)# to gray

# variance = [15,80,250]

# traject = []


# while True:
    
#     suc, frame = cap.read()
#     resized_frame = cv2.resize(frame, None, fx = 0.25, fy = 0.25, interpolation = cv2.INTER_LINEAR)# resize the frame 
#     org_frame = resized_frame.copy()    
#     resized_frame = cv2.GaussianBlur(resized_frame, (7,7), 0)#blur

#     #img_msr = automatedMSRCR(resized_frame , variance)#image enhancement 
#     imgHSV = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2HSV)#tohsv
#     maskr = cv2.inRange(imgHSV, low_green, high_green)#to remove green
    
#     sub = backgroundobject.apply(maskr) #background subtraction
#     sub1 = cv2.cvtColor(sub , cv2.COLOR_BGR2RGB)
#     contours , hi = cv2.findContours(sub, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)# finding contours

#     c = max(contours, key = cv2.contourArea)
#     x, y, width, height = cv2.boundingRect(c)
#     cv2.rectangle(org_frame, (x , y), (x + width, y + height),(0, 0, 255), 2)
# #     M = cv2.moments(c) #getting the centroid
# #     cx = int(M["m10"] / M["m00"])
# #     cy = int(M["m01"] / M["m00"])
            
#     #Calculate optical flow using Lucas-Kanade 
# #     old_pts = np.array([[cx,cy]], dtype="float32").reshape(-1,1,2)
    
# #     new_pts,status,err = cv2.calcOpticalFlowPyrLK(old_gray, sub, old_pts, None, **lk_params)
# #     p0r, _st, _err = cv2.calcOpticalFlowPyrLK(sub, old_gray, new_pts, None, **lk_params)  
    
# #     cv2.circle(org_frame, (int(p0r.ravel()[0]) ,int(p0r.ravel()[1])), 2, (0,255,0), 2)
# #     traject.append((int(p0r.ravel()[0]) ,int(p0r.ravel()[1])))
    
# #     cv2.polylines(org_frame, [np.int32(traject[3:]) for i in traject], False, (0, 255, 0))
        
    
# #     old_gray = sub.copy()
# #     old_pts = p0r.copy()
    
#     #Show Results
#     cv2.imshow('Optical Flow', sub1)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()

In [ ]:
# import pandas as pd
# to store array in ecxel sheet


In [ ]:
# cap = cv2.VideoCapture('F:/Behavoirs/Nice camera angle/1 (5)_Trim.mp4')
# backgroundobject = cv2.createBackgroundSubtractorMOG2(history = 500, varThreshold = 16.5 ,detectShadows=True)

# lower_val = (10, 0, 0)
# upper_val = (135,255,255)

# low_green = np.array([25, 52, 72])
# high_green = np.array([102, 255, 255])

# lk_params = dict(winSize  = (15, 15),
#                 maxLevel = 2,
#                 criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# feature_params = dict(maxCorners = 20,
#                     qualityLevel = 0.3,
#                     minDistance = 10,
#                     blockSize = 7 )


# trajectory_len = 40
# detect_interval = 5
# trajectories = []
# frame_idx = 0

# while True:

#     suc, frame = cap.read()
#     resized_frame = cv2.resize(frame, None, fx=0.3, fy=0.3, interpolation = cv2.INTER_LINEAR)# resize the frame 
#     org_frame = resized_frame.copy()    
#     resized_frame = cv2.GaussianBlur(resized_frame, (7,7), 0)#blur
    
#     imgHSV = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2HSV)#tohsv
#     maskr = cv2.inRange(imgHSV, lower_val, upper_val)#to remove green
    
#     sub = backgroundobject.apply(maskr) #background subtraction
#     contours , hi = cv2.findContours(sub, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)# finding contours
    
#     for contour in contours: 
#         if cv2.contourArea(contour) > 25:
#             x, y, width, height = cv2.boundingRect(contour)
#             cv2.rectangle(org_frame, (x , y), (x + width, y + height),(0, 0, 255), 2)
#             M = cv2.moments(contour)
#             cx = int(M["m10"] / M["m00"])
#             cy = int(M["m01"] / M["m00"])
            
#     #Calculate optical flow for a sparse feature set using the iterative Lucas-Kanade Method
#     if len(trajectories) > 0:
#         img0, img1 = prev_gray, sub
#         p0 = np.float32([trajectory[-1] for trajectory in trajectories]).reshape(-1, 1, 2)
#         p1, _st, _err = cv2.calcOpticalFlowPyrLK(img0, img1, p0, None, **lk_params)
#         p0r, _st, _err = cv2.calcOpticalFlowPyrLK(img1, img0, p1, None, **lk_params)
#         d = abs(p0-p0r).reshape(-1, 2).max(-1)
#         good = d < 1

#         new_trajectories = []

#         # Get all the trajectories
#         for trajectory, (x, y), good_flag in zip(trajectories, p1.reshape(-1, 2), good):
#             if not good_flag:
#                 continue
#             trajectory.append((x, y))
#             if len(trajectory) > trajectory_len:
#                 del trajectory[0]
#             new_trajectories.append(trajectory)
#             # Newest detected point
#             cv2.circle(org_frame, (int(x), int(y)), 2, (0, 0, 255), -1)

#         trajectories = new_trajectories

#         # Draw all the trajectories
#         cv2.polylines(org_frame, [np.int32(trajectory) for trajectory in trajectories], False, (0, 255, 0))
#         cv2.putText(org_frame, 'track count: %d' % len(trajectories), (20, 50), cv2.FONT_HERSHEY_PLAIN, 1, (0,255,0), 2)


#     # Update interval - When to update and detect new features
#     if frame_idx % detect_interval == 0:
#         mask = np.zeros_like(sub)
#         mask[:] = 255

#         # Lastest point in latest trajectory
#         for x, y in [np.int32(trajectory[-1]) for trajectory in trajectories]:
#             cv2.circle(mask, (x, y), 5, 0, -1)

#         # Detect the good features to track
#         p = cv2.goodFeaturesToTrack(sub, mask = mask, **feature_params)
#         if p is not None:
#             # If good features can be tracked - add that to the trajectories
#             for x, y in np.float32(p).reshape(-1, 2):
#                 trajectories.append([(x, y)])


#     frame_idx += 1
#     prev_gray = sub

#     #Show Results
#     cv2.imshow('Optical Flow', org_frame)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()